---
title: "From Cells to Code"
subtitle: "From Excel to Python"
execute:
  echo: true
---

**In this lesson**

- Opening a notebook, and what a cell is
- Fifteen minutes of Python: values, names, lists, and a function you write yourself
- `pandas`: a DataFrame is a sheet, and `read_csv` is File > Open
- `shape`, `head`, `dtypes`, `describe`: the `first look` sheet in four lines
- Every number from chapter 1, recomputed, with the Excel formula beside each line
- The trips-per-day chart in three lines

**The one idea**

> Nothing you learn this chapter is new. Every line of Python here does something you already did in Excel in chapters 1 to 3. The difference is that the Excel version lives in your clicks, and the Python version lives in a file you can run again next quarter, hand to a colleague, or read back in a year to see what you did. Code is a formula bar for the whole analysis.

::: {.content-visible when-format="html"}
[Download this notebook](chapter_04_lecture.ipynb){download="chapter_04_lecture.ipynb"} &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/lshpaner/excel_to_python_exercises/blob/main/notebooks/chapter_04/chapter_04_lecture.ipynb){.colab-link}
:::

::: {.content-visible unless-format="html"}
Notebook and exercises for this chapter: `github.com/lshpaner/excel_to_python_exercises`, folder `notebooks/chapter_04/`.
:::

## Back to the decision

Three chapters of Excel have produced real findings for Metro's board: a thousand trips a day with a holiday dip, two populations of riders with different hours and different bikes, a downtown commute core and a Westside leisure cluster, and an average trip length that needed cleaning before it could be quoted. Every one of those findings is in a workbook. None of them will survive the next quarter's file.

When Metro publishes April to June, someone has to open the new CSV, rebuild the helper columns, rebuild the pivots, redo the `XLOOKUP`s, retype the checks, and hope nothing was missed. The person who does it will usually not be the person who built it. Code solves that problem, and solving it is the only reason this book switches tools halfway.

So this chapter does not add a finding. It takes the first one, the `first look` sheet from chapter 1, and rebuilds it as code. By the end, the same five numbers come out of a notebook that will run on any quarter's file without a click.

## Opening a notebook

A notebook is a document made of cells. A cell holds either text (markdown, like this page) or code. You run a code cell with Shift+Enter, and whatever the last line produces appears underneath it. The cells share memory: a name you define in one cell is available in the next, which is why the order you run them in matters.

Two ways to open one, and the book supports both:

- **Google Colab** (colab.research.google.com): nothing to install, runs in the browser, needs a Google account. Open the chapter's notebook from the companion repository (on the website, the Open in Colab badge at the top of each Python chapter does this in one click), or upload the `.ipynb` from the chapter's release with File > Upload notebook. The first cell downloads the data.
- **Jupyter on your own machine**: install Anaconda, open Anaconda Navigator, launch JupyterLab, open the `.ipynb`. The first cell reads the data from `data/raw/` if it is there, and downloads it if not.

Three things that go wrong in the first ten minutes, and the fix for each:

- **A cell says `NameError: name 'trips' is not defined`.** You ran a cell before the one that creates `trips`. Run the cells from the top (Runtime > Run all in Colab, Kernel > Restart and Run All in Jupyter).
- **The cell shows `[*]` and never finishes.** It is still running, or waiting for input. Press the stop button and run it again.
- **Nothing appears under the cell.** The last line of the cell is an assignment (`x = 5`), and assignments print nothing. Put the name on its own line below it.

## Fifteen minutes of Python

A spreadsheet cell holds a value. Python holds values too, and lets you give them names.

In [ ]:
trips_per_day = 988
quarter = "2026 Q1"
trips_per_day, quarter

`trips_per_day` is a name. `988` is a number. `"2026 Q1"` is text; Python calls it a string, and the quotes are what make it one. The `=` is not Excel's `=`: it does not mean "calculate this", it means "call this that". Read `x = 5` as "x is now 5".

A list is a column.

In [ ]:
durations = [5, 22, 21, 9, 14, 1440]
len(durations), sum(durations), min(durations), max(durations)

That line is `COUNTA`, `SUM`, `MIN`, and `MAX`, in that order. `len` is length. There is no built-in average, because the average is a sum divided by a count and Python expects you to say so:

In [ ]:
sum(durations) / len(durations)

A function is a formula you name once and use many times. Excel gives you `AVERAGE`; Python lets you write it:

In [ ]:
def average(values):
    """Sum divided by count. The same thing AVERAGE does."""
    return sum(values) / len(values)

average(durations)

`def` starts the definition, `values` is the placeholder for whatever you pass in, the indented lines are the body, and `return` is what comes out. The line in triple quotes is a docstring, a note to the next reader about what the function does. Write one for every function; the next reader is usually you.

A comparison gives `True` or `False`, and `True` counts as 1 when you add it. That is how `COUNTIF` works in Python:

In [ ]:
long_trips = [d > 120 for d in durations]
long_trips, sum(long_trips)

The square-bracket expression reads "for each `d` in `durations`, is `d > 120`?" and builds a new list of the answers. Summing it counts the `True`s. Keep that idea; pandas does the same thing on 88,932 rows at once.

That is the Python this chapter needs: names, numbers, strings, lists, a function, a comparison. Everything else is pandas.

## pandas: a DataFrame is a sheet

`pandas` [@mckinney2010] is the library that gives Python a spreadsheet. Its table is called a DataFrame: rows, named columns, one type per column, exactly like the `trips` sheet. `import pandas as pd` loads it, and `pd.read_csv` is File > Open.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
                     "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "figure.titlesize": 11})

LOCAL = "../../data/raw/metro_trips.csv"
URL = ("https://github.com/lshpaner/excel_to_python_exercises/"
       "releases/download/data/metro_trips.csv")

trips = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL)
trips.shape

`shape` is rows by columns: 88,932 trips, 15 columns, the same numbers chapter 1 found by scrolling to the bottom of the sheet. The `LOCAL`/`URL` line reads the file from `data/raw/` when it is there and downloads it from the companion repository when it is not, so the same notebook runs on a laptop and in Colab.

`head` is the top of the sheet:

In [ ]:
trips.head()

`dtypes` is the data dictionary, built by pandas from the data rather than written by hand. `int64` is a whole number, `float64` a decimal, and `str` (or `object` in older pandas) is text. Notice `start_time` is text: pandas does not guess that a column is a date. Chapter 5 fixes that with one line.

In [ ]:
trips.dtypes

`describe` is the `first look` sheet for every numeric column at once:

In [ ]:
trips["duration"].describe().round(1)

Read it against chapter 1: count 88,932, mean 32.9, median (the `50%` row) 14, min 1, max 1,440. The `25%` and `75%` rows are new: a quarter of trips are 7 minutes or under, three quarters are 27 or under. Chapter 7 will call that spread, and it is the reason the mean and median disagree.

## The first look, recomputed

Here is chapter 1's `first look` sheet, one Excel formula per line, with the pandas equivalent run beside it. `trips["duration"]` is a column, and a column has the same five questions a range does.

In [ ]:
d = trips["duration"]

first_look = pd.DataFrame({
    "Excel":  ["=COUNTA(B:B)", "=AVERAGE(B:B)", "=MEDIAN(B:B)", "=MIN(B:B)", "=MAX(B:B)",
               '=COUNTIF(B:B, ">120")', '=COUNTIF(B:B, ">120") / COUNTA(B:B)'],
    "pandas": ["d.count()", "d.mean()", "d.median()", "d.min()", "d.max()",
               "(d > 120).sum()", "(d > 120).mean()"],
    "value":  [d.count(), d.mean(), d.median(), d.min(), d.max(),
               (d > 120).sum(), (d > 120).mean()],
})
first_look["value"] = [f"{v:,.3f}".rstrip("0").rstrip(".") for v in first_look["value"]]
first_look

Two of these deserve a second look.

`(d > 120).sum()` is `COUNTIF`. `d > 120` compares every row at once and gives a column of `True`/`False`; summing it counts the `True`s, the same trick as the list earlier. 2,844 trips over two hours.

`(d > 120).mean()` is the share. The mean of a `True`/`False` column is the fraction that are `True`, because `True` is 1. 3.2%. No separate division, no second cell to keep in sync.

The yellow input cell from chapter 1 (type a pass type, four rows update) is a function:

In [ ]:
def pass_summary(df, pass_type):
    """Count, average and median duration for one passholder type."""
    sub = df[df["passholder_type"] == pass_type]
    return pd.Series({"trips": len(sub), "avg_minutes": sub["duration"].mean(),
                      "median_minutes": sub["duration"].median()})

pass_summary(trips, "Walk-up").round(1)

`df[df["passholder_type"] == pass_type]` is a filter: keep the rows where the comparison is `True`. That is Excel's filter dropdown, written down. Change `"Walk-up"` to `"Monthly Pass"` and run it again; that is the yellow cell.

In [ ]:
#| echo: false
pass_summary(trips, "Monthly Pass").round(1)

Walk-up riders average 67 minutes, Monthly Pass holders 19. Same finding as chapter 1, now in a function that will give the same answer on any quarter's file.

## The chart, in three lines

Chapter 1 ended with trips per day. In pandas: turn `start_time` into a real date, count rows per date, plot.

In [ ]:
#| label: fig-daily-pandas
#| fig-cap: "Trips per day, January to March 2026, from three lines of pandas. Compare with the chapter 1 chart built from a COUNTIFS column."
trips["start_time"] = pd.to_datetime(trips["start_time"])
per_day = trips.groupby(trips["start_time"].dt.date).size()
ax = per_day.plot(figsize=(9, 3.6), color="#1F4E79", linewidth=1.8)
ax.set_ylim(0, per_day.max() * 1.1); ax.set_xlabel(""); ax.set_ylabel("Trips")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

Line one: `pd.to_datetime` turns the text column into dates, which is what the `TEXT` and `HOUR` helper columns were working around in chapter 2. Line two: `groupby(...).size()` is the `COUNTIFS` column on the `trips per day` sheet, 90 formulas replaced by one. Line three draws it. The rest is cosmetics.

The Presidents' Day dip on February 16 is there, 238 trips, same as the sheet. It had better be: same data, same count. If a number in Python disagrees with the Excel version, one of them is wrong, and from chapter 5 on, checking the two against each other is how you will know your code is right.

## Back to the decision

Nothing new for the board this chapter, by design. What changed is what happens when the next quarter arrives: point `read_csv` at the new file and every number on this page updates. The checks from chapter 3, the pivots from chapter 2, the joins: chapter 5 rebuilds all of them the same way, and then the whole Excel analysis is a notebook that runs in under a minute.

The habit to take from this chapter: when you write a line of pandas, say the Excel formula it replaces out loud. If you cannot, you do not yet know what the line does.

## Looking ahead

- **Worked exercise:** `chapter_04_guided.ipynb`. Open the notebook, load the data, reproduce the `first look` sheet, and write one function. Every cell has the Excel formula it replaces in a comment above it.
- **On your own:** `chapter_04_solo.ipynb`. The data dictionary from `dtypes`, five questions answered with one line each, one function with a docstring, one chart.
- **Before the next chapter:** reread the `by weekday` and `two ways` sheets from chapter 2. Chapter 5 rebuilds both with `groupby`, and it goes faster if the Excel version is fresh.

## Appendix: reproducing this chapter {.unnumbered}

The companion notebook, `chapter_04_lecture.ipynb`, is generated from this page by `make notebooks_chapter_04` (`quarto convert`), so the two cannot drift. The exercise notebooks come from `tools/build_chapter_04_exercises.py`, which writes the reader and solution versions from one source. Data loads from `data/raw/metro_trips.csv` when present and from the companion repository's `data` release otherwise.